# SCOPE — PTXPhish transaction-risk model (reproducible pipeline)

This notebook performs the full data-to-model workflow. It does **not** assume a target accuracy. It extracts payload-level transaction features from labelled PTXPhish hashes, checks coverage/labels, evaluates on a stratified holdout, reports minority-class metrics, and exports a versioned model bundle.

**Important:** transaction hashes are identifiers, not predictive features. Raw addresses and hashes are deliberately excluded from the feature matrix to reduce memorization. This is a research baseline, not a production security guarantee.


In [ ]:
from pathlib import Path
import json, time, math, os, random
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, average_precision_score, roc_auc_score)
from sklearn.base import clone
import joblib

SEED = 42
ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'ptxphish_labeled_transaction_hashes.csv'
OUT = ROOT / 'artifacts'
CACHE = ROOT / 'data' / 'tx_feature_cache.csv'
OUT.mkdir(exist_ok=True)
print('Working directory:', ROOT)
print('Input exists:', DATA_PATH.exists())


## 1. Load the audited labels

In [ ]:
labels = pd.read_csv(DATA_PATH)
required = {'tx_hash','label','source_category'}
missing = required - set(labels.columns)
if missing: raise ValueError(f'Missing required columns: {missing}')
labels['tx_hash'] = labels['tx_hash'].astype(str).str.lower().str.strip()
labels = labels.drop_duplicates('tx_hash').copy()
labels = labels[labels['tx_hash'].str.fullmatch(r'0x[a-f0-9]{64}', na=False)].copy()
labels['label'] = labels['label'].astype(str).str.lower()
labels = labels[labels['label'].isin(['benign','phishing'])].copy()
labels['y'] = (labels['label'] == 'phishing').astype(int)
print('Rows:', len(labels), '| unique hashes:', labels.tx_hash.nunique())
print(labels['label'].value_counts())
print('Category count:', labels.source_category.nunique())
if labels['y'].nunique() != 2: raise ValueError('Need both benign and phishing labels.')


## 2. Fetch public Ethereum transaction + receipt data (resumable)

The default endpoint is a public RPC and may throttle requests. For a smoke test set `MAX_TRANSACTIONS = 100`; for the full run leave it as `None`. Existing cache rows are reused. The notebook does not use transaction hash as a model feature.

In [ ]:
RPC_URL = os.environ.get('ETH_RPC_URL', 'https://ethereum-rpc.publicnode.com')
MAX_TRANSACTIONS = None  # None = all rows; use 100 only to test RPC connectivity
BATCH_SIZE = 100
WORKERS = 6
TIMEOUT_SECONDS = 20

work = labels.copy()
if MAX_TRANSACTIONS is not None: work = work.head(int(MAX_TRANSACTIONS)).copy()
if CACHE.exists():
    cache = pd.read_csv(CACHE)
    done = set(cache.get('tx_hash', pd.Series(dtype=str)).astype(str).str.lower())
else:
    cache = pd.DataFrame()
    done = set()
queue = work[~work.tx_hash.isin(done)]
print('Labelled rows in this run:', len(work), '| cached:', len(done), '| to fetch:', len(queue))

def rpc(method, params):
    payload={'jsonrpc':'2.0','id':1,'method':method,'params':params}
    r=requests.post(RPC_URL, json=payload, timeout=TIMEOUT_SECONDS)
    r.raise_for_status(); obj=r.json()
    if obj.get('error'): raise RuntimeError(str(obj['error']))
    return obj.get('result')

def hxint(x):
    try: return int(x,16) if x else 0
    except Exception: return 0

def get_features(row):
    h=row.tx_hash
    err=''
    try:
        tx=rpc('eth_getTransactionByHash',[h])
        if not tx: return {'tx_hash':h,'fetch_ok':0,'fetch_error':'transaction_not_found'}
        receipt=rpc('eth_getTransactionReceipt',[h])
        data=(tx.get('input') or '0x').lower()
        to=(tx.get('to') or '').lower()
        value=hxint(tx.get('value'))
        gas=hxint(tx.get('gas'))
        gp=hxint(tx.get('gasPrice'))
        selector=data[:10] if len(data)>=10 else '0x'
        input_bytes=max(0,(len(data)-2)//2)
        # Generic EVM payload features; no raw tx hash or raw address enters model.
        if selector in ('0xa9059cbb','0x23b872dd'): kind='transfer'
        elif selector in ('0x095ea7b3','0xd505accf','0x8fcbaf0c'): kind='approval'
        elif selector=='0x': kind='native_or_empty'
        else: kind='contract_call'
        return {'tx_hash':h,'fetch_ok':1,'fetch_error':'','label':row.label,'y':int(row.y),
          'source_category':row.source_category,'selector':selector,'selector_kind':kind,
          'input_bytes':input_bytes,'has_input':int(input_bytes>0),'value_eth':value/1e18,
          'gas_limit':gas,'gas_price_gwei':gp/1e9,'is_contract_creation':int(not to),
          'receipt_status':hxint((receipt or {}).get('status')),'receipt_logs':len((receipt or {}).get('logs') or []),
          'tx_type':hxint(tx.get('type')),'block_number':hxint(tx.get('blockNumber'))}
    except Exception as e:
        return {'tx_hash':h,'fetch_ok':0,'fetch_error':type(e).__name__+': '+str(e)[:180]}

new_rows=[]
items=list(queue.itertuples(index=False))
for start in range(0,len(items),BATCH_SIZE):
    batch=items[start:start+BATCH_SIZE]
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futures=[ex.submit(get_features,r) for r in batch]
        for f in as_completed(futures): new_rows.append(f.result())
    # Save checkpoint after each batch so interrupted runs can resume.
    fresh=pd.DataFrame(new_rows)
    if CACHE.exists(): old=pd.read_csv(CACHE); merged=pd.concat([old,fresh],ignore_index=True)
    else: merged=fresh
    merged=merged.drop_duplicates('tx_hash',keep='last')
    merged.to_csv(CACHE,index=False)
    print(f'Checkpoint {min(start+BATCH_SIZE,len(items))}/{len(items)}; cached rows={len(merged)}')
    time.sleep(0.15)

features = pd.read_csv(CACHE) if CACHE.exists() else pd.DataFrame()
print('Cached rows:',len(features))
if len(features):
    print('Fetch success:',int(features.fetch_ok.fillna(0).sum()),'/',len(features))
    display(features.head())


## 3. Data quality gate

Do not train on failed lookups or if the feature coverage is too low. A limited smoke test is not a final training dataset.

In [ ]:
if features.empty: raise RuntimeError('No feature rows. Check RPC connectivity and rerun this cell after fetching.')
features['tx_hash']=features['tx_hash'].astype(str).str.lower()
valid=features[features['fetch_ok'].fillna(0).astype(int)==1].copy()
valid=valid.merge(labels[['tx_hash','label','y','source_category']],on='tx_hash',how='inner',suffixes=('','_label'))
# Labels in the source CSV are authoritative; overwrite cached label fields.
valid['label']=valid['label_label']; valid['y']=valid['y_label']; valid['source_category']=valid['source_category_label']
valid=valid.drop(columns=[c for c in ['label_label','y_label','source_category_label'] if c in valid.columns])
print('Valid fetched rows:',len(valid),'of',len(labels))
print('Coverage:',round(len(valid)/max(1,len(labels))*100,2),'%')
print(valid.groupby('label').size())
print('Selector kinds by label:')
display(pd.crosstab(valid['selector_kind'],valid['label'],normalize='columns').round(3))
if len(valid)<1000: raise RuntimeError('Too few successfully fetched examples for a credible model. Resolve RPC coverage first; do not treat a small smoke test as final.')
if valid['y'].nunique()!=2: raise RuntimeError('Only one class remains after fetching; cannot train.')


## 4. Train and compare candidates once

Split by transaction row, stratified by label. This is a baseline evaluation; transactions may be related, so also inspect selector overlap and do not claim cross-chain generalization from this split.

In [ ]:
NUMERIC=['input_bytes','has_input','value_eth','gas_limit','gas_price_gwei','is_contract_creation','receipt_status','receipt_logs','tx_type']
CATEGORICAL=['selector','selector_kind']
FEATURES=NUMERIC+CATEGORICAL
X=valid[FEATURES].copy(); y=valid['y'].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=SEED,stratify=y)
pre=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),NUMERIC),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',min_frequency=5))]),CATEGORICAL),
])
candidates={
 'ExtraTrees':ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced',random_state=SEED,n_jobs=-1),
 'RandomForest':RandomForestClassifier(n_estimators=250,min_samples_leaf=2,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1),
}
results=[]; fitted={}
for name,est in candidates.items():
    pipe=Pipeline([('preprocess',clone(pre)),('model',est)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); proba=pipe.predict_proba(X_test)[:,1]
    p,r,f,_=precision_recall_fscore_support(y_test,pred,average='binary',zero_division=0)
    results.append({'model':name,'accuracy':accuracy_score(y_test,pred),'phishing_precision':p,'phishing_recall':r,'phishing_f1':f,
      'phishing_average_precision':average_precision_score(y_test,proba),
      'roc_auc':roc_auc_score(y_test,proba) if y_test.nunique()==2 else np.nan})
    fitted[name]=(pipe,proba,pred)
results_df=pd.DataFrame(results).sort_values(['phishing_f1','phishing_recall'],ascending=False)
display(results_df)


## 5. Evaluate the selected model and check selector generalization

In [ ]:
best_name=str(results_df.iloc[0]['model'])
best,proba,pred=fitted[best_name]
print('Selected:',best_name)
print(classification_report(y_test,pred,target_names=['benign','phishing'],digits=4,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['benign','phishing'],values_format='d')
plt.title('Held-out confusion matrix'); plt.show()
print('Test selector overlap with train:',round(len(set(X_test.selector)&set(X_train.selector))/max(1,len(set(X_test.selector)))*100,2),'%')
print('NOTE: high selector overlap can make the random holdout optimistic. Validate on a later/time-based or unseen-contract holdout before production.')


## 6. Export a versioned model bundle and run a parity smoke test

The saved bundle records the feature schema. SCOPE integration must supply the same transaction-payload features; the existing simplified action schema alone is insufficient. Deterministic SCOPE `DENY` always wins.

In [ ]:
from datetime import datetime, timezone
metrics=results_df.to_dict(orient='records')
bundle={
 'model':best,
 'pipeline':best,
 'feature_columns':FEATURES,
 'numeric_columns':NUMERIC,
 'categorical_columns':CATEGORICAL,
 'positive_label':'phishing',
 'training_dataset':'PTXPhish labelled transaction hashes + Ethereum RPC payload features',
 'trained_at_utc':datetime.now(timezone.utc).isoformat(),
 'train_rows':int(len(X_train)),'test_rows':int(len(X_test)),
 'metrics':metrics,
 'policy':'advisory_only; deterministic SCOPE DENY cannot be overridden',
 'chain_scope':'Ethereum mainnet transaction payloads; cross-chain validation required'
}
joblib.dump(bundle,OUT/'scope_ptxphish_risk_model.joblib')
report={'selected_model':best_name,'metrics':metrics,'valid_fetched_rows':int(len(valid)),'label_counts':valid['label'].value_counts().to_dict(),
 'feature_columns':FEATURES,'warnings':['Not production-validated','Ethereum-only source data','No 98-99% performance promise','Action schema must expose same payload features','Deterministic deny must remain authoritative']}
(OUT/'evaluation_report.json').write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
# Reload check ensures the artifact is readable and has the same interface.
loaded=joblib.load(OUT/'scope_ptxphish_risk_model.joblib')
smoke=loaded['pipeline'].predict_proba(X_test.iloc[:3])[:,1]
print('Artifact saved:',(OUT/'scope_ptxphish_risk_model.joblib').resolve())
print('Report saved:',(OUT/'evaluation_report.json').resolve())
print('Reload smoke scores (phishing probability):',np.round(smoke,4).tolist())


## 7. SCOPE integration checklist

- Do not wire the artifact into production until data coverage and minority-class metrics are reviewed.
- Extend the API's internal evaluation path to pass `input`/calldata-derived features through the exact same feature builder. Do not accept client-supplied model features as trusted facts.
- If calldata/payload is absent, malformed, or from an unsupported chain, mark ML unavailable and use existing deterministic rules.
- Preserve hard `DENY` decisions before and after ML scoring. The ML score can add a warning/consent requirement, never relax a hard deny.
- Add regression tests for missing features, unseen selector, unlimited approval, large amount, unsupported chain, and model artifact version.
- PTXPhish labels/categories are not proof of future generalization; validate against a separate later-time and chain-specific dataset before deployment.
